## GetONCRawData – Incremental BPR Parquet Pipeline

Fetches raw hex data from the ONC API **one day at a time** (2022-05-23 → today),
parses each ASCII hex line into frequency counts and periods, and appends everything
to a partitioned Parquet dataset at `out/NCHR_BPR_raw.parquet/`.

Rerun at any time to top-up the dataset with the latest days — already-stored days are skipped automatically.

> **Kernel:** select the `bpr-nchr` conda environment.  
> **Auth:** set the `ONC_TOKEN` environment variable (see https://oceannetworkscanada.github.io/api-python-client/ )

In [ ]:
import os
import logging
from dotenv import load_dotenv

load_dotenv()   # loads .env from the project root (safe no-op if file absent)

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

token = os.getenv('ONC_TOKEN')
if not token:
    raise EnvironmentError(
        'ONC_TOKEN is not set. Copy .env.example → .env and add your token, '
        'or set the variable in your shell before launching Jupyter.'
    )

In [2]:
from onc import ONC

onc = ONC(token)

In [3]:
from src.build_parquet import run_incremental_build, load_dataset

INFO numexpr.utils: NumExpr defaulting to 16 threads.


In [4]:
# ── Configuration ────────────────────────────────────────────────────────────
LOCATION_CODE        = 'NCHR'
DEVICE_CATEGORY_CODE = 'BPR'
START_DATE           = '2022-05-23'   # earliest date to fetch if dataset is empty
# ─────────────────────────────────────────────────────────────────────────────

In [5]:
from datetime import datetime, timezone

run_incremental_build(
    onc_client=onc,
    start_date=datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc),
    location_code=LOCATION_CODE,
    device_category_code=DEVICE_CATEGORY_CODE,
)

WARNING src.build_parquet: Could not read existing dataset: No match for FieldRef.Name(dmas_time) in __fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string
INFO src.build_parquet: Starting fresh from 2022-05-23
INFO src.build_parquet: Starting fresh from 2022-05-23


Fetching 1402 day(s) from 2022-05-23 to 2026-03-24 …


  0%|          | 4/1402 [00:10<59:06,  2.54s/day]  WARNING src.parse_hex: Failed to parse 1 / 86385 lines
WARNING src.parse_hex: Failed to parse 1 / 86385 lines
  1%|          | 16/1402 [00:41<1:00:19,  2.61s/day]WARNING src.parse_hex: Failed to parse 216 / 83657 lines
WARNING src.parse_hex: Failed to parse 216 / 83657 lines
  2%|▏         | 22/1402 [00:57<1:01:06,  2.66s/day]WARNING src.parse_hex: Failed to parse 324 / 52483 lines
WARNING src.parse_hex: Failed to parse 324 / 52483 lines
  2%|▏         | 23/1402 [00:59<53:22,  2.32s/day]  WARNING src.parse_hex: Failed to parse 4666 / 24155 lines
WARNING src.parse_hex: Failed to parse 4666 / 24155 lines
  5%|▍         | 65/1402 [02:46<58:10,  2.61s/day]  WARNING src.parse_hex: Failed to parse 1 / 86385 lines
WARNING src.parse_hex: Failed to parse 1 / 86385 lines
  5%|▌         | 74/1402 [03:09<59:28,  2.69s/day]WARNING src.parse_hex: Failed to parse 1 / 86364 lines
WARNING src.parse_hex: Failed to parse 1 / 86364 lines
 17%|█▋        | 

Done. Days written: 1402 | empty: 0 | errors: 0
Dataset: /Users/schlesin/Library/CloudStorage/GoogleDrive-schlesin@oceannetworks.ca/My Drive/ONC Projects/CORK_Stuff/Endeavou_calibration_Experiment_MattWei_2026/out/NCHR_BPR_raw.parquet


In [6]:
# Preview the full dataset
df = load_dataset()
print(f'Total rows : {len(df):,}')
print(f'Date range : {df.index.min()}  →  {df.index.max()}')
df.head()

Total rows : 4,060,813
Date range : 2022-05-31 00:00:00.702000  →  2026-03-24 23:59:59.055000


,ppc_time,t_housing_counts,xFT,xFP,X_period_us,T_period_us,year,month
dmas_time,,,,,,,,
2022-05-31 00:00:00.702,2022-05-31 00:00:03,12933627,702069311,1789632654,5.817316,28.333626,2022,5
2022-05-31 00:00:01.703,2022-05-31 00:00:04,12933778,702069307,1789632689,5.817316,28.333627,2022,5
2022-05-31 00:00:02.703,2022-05-31 00:00:05,12933741,702069312,1789632606,5.817316,28.333626,2022,5
2022-05-31 00:00:03.703,2022-05-31 00:00:06,12933592,702069307,1789632611,5.817316,28.333626,2022,5
2022-05-31 00:00:04.703,2022-05-31 00:00:07,12933806,702069309,1789632620,5.817316,28.333626,2022,5


In [ ]:
# Optional: load a specific date range
# df_sub = load_dataset(date_from='2025-01-01', date_to='2025-02-01')
# df_sub.head()